# HYROX by the numbers

What actually separates HYROX athletes, where the time goes at each level, and which weaknesses really close when people race again.

Built on public HYROX results via [pyrox-client](https://github.com/vmatei2/pyrox-client) (MIT licence), with athlete names removed. Independent analysis, not affiliated with HYROX. Companion app: **Next Five** — see where *your* next 5 minutes are.

**How to read this:** each section asks one question, tests it, and prints a one-line answer (➡️). Results data shows *associations*, not causes, so findings are phrased as "consistent with", not "proves".

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

def find(name):
    for pattern in [f"/kaggle/input/**/{name}", f"kaggle/{name}", f"data/{name}", name]:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    return None

RUNS = [f"run{i}_time" for i in range(1, 9)]
STATIONS = ["skiErg_time", "sledPush_time", "sledPull_time", "burpeeBroadJump_time",
            "rowErg_time", "farmersCarry_time", "sandbagLunges_time", "wallBalls_time"]
SEGS = RUNS + STATIONS + ["roxzone_time"]
LABEL = {**{f"run{i}_time": f"Run {i}" for i in range(1, 9)},
         "skiErg_time": "SkiErg", "sledPush_time": "Sled Push", "sledPull_time": "Sled Pull",
         "burpeeBroadJump_time": "Burpees", "rowErg_time": "Row", "farmersCarry_time": "Farmers Carry",
         "sandbagLunges_time": "Lunges", "wallBalls_time": "Wall Balls", "roxzone_time": "Roxzone"}
STRENGTH = ["sledPush_time", "sledPull_time", "farmersCarry_time", "sandbagLunges_time"]
ENGINE = ["skiErg_time", "rowErg_time", "burpeeBroadJump_time", "wallBalls_time"]
RACE = ["season", "location", "year", "gender", "division"]

FINDINGS = []
def finding(text):
    FINDINGS.append(text)
    print("➡️", text)

raw = pd.read_parquet(find("hyrox_results_anonymised.parquet"))
print(f"{len(raw):,} results · {raw.groupby(['season','location','year']).ngroups} race editions · seasons {sorted(raw.season.unique())}")

## 0. Cleaning and a fair way to compare levels

We keep individual Open and Pro results with every split present, and drop results where the splits don't add up to the finish time (timing errors).

Comparing raw finish times across races is unfair: venues differ, and men's and women's times differ. So an athlete's **level** is their percentile *within their own race, gender and division*. "Top 10%" means the same thing in Mumbai and London.

In [ ]:
df = raw[raw["division"].astype(str).isin(["open", "pro"])].dropna(subset=SEGS + ["total_time"]).copy()
gap = (df[SEGS].sum(axis=1) - df["total_time"]).abs() / df["total_time"]
before = len(df)
df = df[gap < 0.03]
print(f"Kept {len(df):,} of {before:,} complete results ({before - len(df):,} dropped: splits didn't add up)")

df["pct"] = df.groupby(RACE, observed=True)["total_time"].rank(pct=True)
df["level"] = pd.cut(df["pct"], [0, .10, .25, .50, .75, 1.0],
                     labels=["Top 10%", "10–25%", "25–50%", "50–75%", "Bottom 25%"])
df["run_total"] = df[RUNS].sum(axis=1)
df["station_total"] = df[STATIONS].sum(axis=1)
LEVELS = list(df["level"].cat.categories)
df.groupby("level", observed=True)["total_time"].median().rename("median finish (min)").round(1).to_frame()

## 1. Where does the time go at each level?

Research on a simulated race found running was the largest share of total time. Does that share change as you move down the field?

In [ ]:
share = df.groupby("level", observed=True)[["run_total", "station_total", "roxzone_time"]].sum()
share = share.div(share.sum(axis=1), axis=0) * 100
share.columns = ["Running", "Stations", "Roxzone"]
share.plot(kind="barh", stacked=True, color=["#3b6fd4", "#e0a526", "#2e9d5b"])
plt.gca().invert_yaxis(); plt.xlabel("% of race time"); plt.title("Share of race time by level"); plt.legend(loc="lower right"); plt.show()
display(share.round(1))
top, bottom = share.loc[LEVELS[0]], share.loc[LEVELS[-1]]
finding(f"Running is {top['Running']:.0f}% of race time for the top 10% and {bottom['Running']:.0f}% for the bottom 25%; "
        f"stations go from {top['Stations']:.0f}% to {bottom['Stations']:.0f}%.")

## 2. What separates each level from the next?

For each step up (e.g. 50–75% → 25–50%), which segments account for the difference? This is where a minute is "cheapest" at your level.

In [ ]:
med = df.groupby("level", observed=True)[SEGS].median()
rows = []
for slower, faster in zip(LEVELS[::-1][:-1], LEVELS[::-1][1:]):
    diff = (med.loc[slower] - med.loc[faster])
    runs = diff[RUNS].sum(); total = diff.sum()
    top3 = diff.drop(RUNS).nlargest(3)
    rows.append({"step": f"{slower} → {faster}", "total gap (min)": round(total, 1),
                 "from running": f"{runs / total:.0%}",
                 "biggest non-run gaps": ", ".join(f"{LABEL[k]} {v*60:.0f}s" for k, v in top3.items())})
steps = pd.DataFrame(rows); display(steps)
first = rows[0]
finding(f"Moving from the bottom quarter to the 50–75% band is a {first['total gap (min)']} min gap, "
        f"{first['from running']} of it running; biggest non-run gaps: {first['biggest non-run gaps']}.")

## 3. Compromised running: which station hurts the next run most?

Each run follows a station. We compare every run with Run 1 (fresh legs).

In [ ]:
ratio = df[RUNS].div(df["run1_time"], axis=0)
ratio["level"] = df["level"]
by_level = ratio.groupby("level", observed=True)[RUNS[1:]].median()
after = {"run2_time": "after SkiErg", "run3_time": "after Sled Push", "run4_time": "after Sled Pull",
         "run5_time": "after Burpees", "run6_time": "after Row", "run7_time": "after Farmers",
         "run8_time": "after Lunges"}
for lvl in by_level.index:
    plt.plot([after[c] for c in RUNS[1:]], (by_level.loc[lvl] - 1) * 100, marker="o", label=lvl)
plt.axhline(0, color="grey", lw=.8); plt.ylabel("% slower than Run 1"); plt.xticks(rotation=30, ha="right")
plt.title("How much slower each run is than Run 1"); plt.legend(); plt.show()
overall = (ratio[RUNS[1:]].median() - 1) * 100
worst = overall.idxmax()
finding(f"The run {after[worst]} is the slowest relative to Run 1 (+{overall[worst]:.0f}% on average). "
        f"Top 10% slow by {(by_level.loc[LEVELS[0]].mean()-1)*100:.0f}% across runs, bottom 25% by {(by_level.loc[LEVELS[-1]].mean()-1)*100:.0f}%.")

## 4. Do the later runs separate athletes more?

It has been reported that runs 5–8 correlate most strongly with finish time. We test it: how strongly does each segment track an athlete's overall level?

In [ ]:
rel = df[SEGS].div(df.groupby(RACE, observed=True)[SEGS].transform("median"))
corr = rel.corrwith(df["pct"], method="spearman").rename(LABEL).sort_values()
corr.plot(kind="barh", color=["#3b6fd4" if k.startswith("Run") else "#e0a526" for k in corr.index])
plt.xlabel("Correlation with overall level (higher = separates athletes more)"); plt.title("Which segments track overall level?"); plt.show()
early = corr[[f"Run {i}" for i in range(1, 5)]].mean(); late = corr[[f"Run {i}" for i in range(5, 9)]].mean()
verdict = "supports" if late > early else "does not support"
finding(f"Late runs (5–8) correlate {late:.2f} with level vs {early:.2f} for early runs (1–4): {verdict} the late-runs hypothesis. "
        f"Strongest single segment: {corr.idxmax()}.")

## 5. Do strength stations cost slower athletes disproportionately?

A 2026 study of **Pro and Elite** races found strength-determined stations cost slower performers disproportionately more time. Does that hold in the much bigger **Open** field?

In [ ]:
open_ = df[df["division"].astype(str) == "open"]
m = open_.groupby("level", observed=True)[SEGS].median()
spread = (m.loc[LEVELS[-1]] / m.loc[LEVELS[0]])
groups = {"Strength stations": STRENGTH, "Engine/skill stations": ENGINE, "Runs": RUNS}
g = pd.Series({k: spread[v].mean() for k, v in groups.items()})
spread.rename(LABEL).sort_values().plot(kind="barh", color="#e0a526")
plt.xlabel("Bottom 25% time ÷ top 10% time"); plt.title("How much slower the back of the field is, per segment (Open)"); plt.show()
display(g.round(2).rename("bottom ÷ top").to_frame())
finding(f"In Open, the bottom 25% take {g['Strength stations']:.1f}× as long as the top 10% on strength stations, "
        f"vs {g['Engine/skill stations']:.1f}× on engine/skill stations and {g['Runs']:.1f}× on runs.")

## 6. Does going out fast backfire?

**Start** = Run 1 compared with the athlete's typical run (runs 2–8). **Fade** = late runs (6–8) compared with early runs (2–4).

In [ ]:
start = df["run1_time"] / df[RUNS[1:]].median(axis=1)
fade = df[["run6_time", "run7_time", "run8_time"]].mean(axis=1) / df[["run2_time", "run3_time", "run4_time"]].mean(axis=1)
q = pd.qcut(start, 5, labels=["Fastest start", "Fast", "Average", "Slow", "Most conservative"])
t = pd.DataFrame({"fade %": (fade.groupby(q, observed=True).median() - 1) * 100,
                  "median level (percentile)": df["pct"].groupby(q, observed=True).median() * 100}).round(1)
display(t)
diff = t["fade %"].iloc[0] - t["fade %"].iloc[-1]
finding(f"Fastest starters fade {t['fade %'].iloc[0]:.1f}% in late runs vs {t['fade %'].iloc[-1]:.1f}% for the most conservative "
        f"({diff:+.1f} points). Consistent with {'going out hard costing you later' if diff > 0 else 'no clear cost to a fast start'}.")

## 7. Which venues are fast or slow?

Caution: a venue's median also reflects *who* races there, not just the course.

In [ ]:
v = (df[df["division"].astype(str) == "open"]
     .groupby(["location", "year"], observed=True)
     .agg(finishers=("total_time", "size"), median_finish=("total_time", "median"),
          roxzone=("roxzone_time", "median"), sled_push=("sledPush_time", "median")))
v = v[v["finishers"] >= 200]
if len(v) >= 5:
    v.index = [f"{l.title()} {y}" for l, y in v.index]
    both = pd.concat([v.nsmallest(8, "median_finish"), v.nlargest(8, "median_finish")])
    both["median_finish"].sort_values().plot(kind="barh", color="#3b6fd4")
    plt.xlabel("Median Open finish (min)"); plt.title("Fastest and slowest venues"); plt.show()
    r = v["roxzone"].corr(v["median_finish"])
    finding(f"Across {len(v)} venues, median Open finish ranges from {v.median_finish.min():.0f} to {v.median_finish.max():.0f} min; "
            f"venue roxzone time correlates {r:.2f} with venue finish time.")
else:
    print("Not enough large venues for this section.")

## 8. What declines most with age?

In [ ]:
if "age_group" in df.columns and df["age_group"].notna().any():
    ages = df.dropna(subset=["age_group"]).copy()
    ages["age_start"] = pd.to_numeric(ages["age_group"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    ages = ages[ages["division"].astype(str) == "open"].dropna(subset=["age_start"])
    young = ages[ages["age_start"].between(25, 34)]
    old = ages[ages["age_start"] >= 50]
    if len(young) > 100 and len(old) > 100:
        decline = (old[SEGS].median() / young[SEGS].median() - 1) * 100
        decline.rename(LABEL).sort_values().plot(kind="barh", color="#2e9d5b")
        plt.xlabel("% slower: age 50+ vs age 25–34"); plt.title("Which segments slow most with age (Open)"); plt.show()
        grp = {k: decline[v].mean() for k, v in groups.items()}
        finding(f"Athletes 50+ are {grp['Runs']:.0f}% slower on runs, {grp['Strength stations']:.0f}% on strength stations and "
                f"{grp['Engine/skill stations']:.0f}% on engine/skill stations than 25–34s. Most age-sensitive: {decline.rename(LABEL).idxmax()}.")
    else:
        print("Not enough athletes in the age groups compared.")
else:
    print("No age-group column in this dataset.")

## 9. Athlete types: runners, lifters, or balanced?

We describe each athlete's *shape*: how their runs, strength stations and engine stations compare with the field, relative to their own overall level. Then k-means groups similar shapes.

In [ ]:
prof = pd.DataFrame({
    "runs": rel[RUNS].mean(axis=1), "strength": rel[STRENGTH].mean(axis=1),
    "engine": rel[ENGINE].mean(axis=1), "roxzone": rel["roxzone_time"]})
shape = prof.div(prof.mean(axis=1), axis=0)          # remove overall level: keep only the shape
X = ((shape - shape.mean()) / shape.std()).to_numpy()

rng = np.random.default_rng(0)
centres = X[rng.choice(len(X), 3, replace=False)]
for _ in range(50):                                   # k-means, written out so it's transparent
    labels = np.argmin(((X[:, None, :] - centres[None]) ** 2).sum(-1), axis=1)
    centres = np.array([X[labels == k].mean(0) for k in range(3)])

types = shape.groupby(labels).mean()
NAMES = {"runs": "Runner", "strength": "Strong", "engine": "Engine", "roxzone": "Efficient"}
used, names = set(), []
for _, row in types.iterrows():                       # name each group by its relative strength
    if row.max() - row.min() < 0.03:
        n = "Balanced"
    else:
        n = next(NAMES[k] for k in row.sort_values().index if NAMES[k] not in used)
    used.add(n); names.append(n)
types.index = names
types["share of athletes"] = pd.Series(labels).value_counts(normalize=True).sort_index().values
types["median level"] = df["pct"].groupby(labels).median().values
display(types.round(2))
finding("Athlete types: " + "; ".join(f"{i} = {r['share of athletes']:.0%} of athletes" for i, r in types.iterrows()) + ".")

## 10. India vs the global field

**Level-matched:** we compare athletes who finished in the *same time band* (5-minute bins, same gender), so this shows differences in race *profile*, not in overall speed.

In [ ]:
INDIA = {"mumbai", "delhi", "new delhi", "bengaluru", "bangalore", "hyderabad", "chennai"}
df["india"] = df["location"].astype(str).str.lower().isin(INDIA)
if df["india"].sum() >= 100:
    df["bin"] = (df["total_time"] // 5) * 5
    diffs = []
    for (g, b), grp in df.groupby(["gender", "bin"], observed=True):
        ind, rest = grp[grp["india"]], grp[~grp["india"]]
        if len(ind) >= 10 and len(rest) >= 50:
            d = (ind[SEGS].median() / rest[SEGS].median() - 1) * 100
            diffs.append(d * len(ind))
    if diffs:
        weighted = sum(diffs) / df.loc[df["india"]].shape[0]
        weighted = weighted.rename(LABEL).sort_values()
        weighted.plot(kind="barh", color=["#2e9d5b" if x < 0 else "#d44b3b" for x in weighted])
        plt.xlabel("% vs global athletes with the same finish time (− = faster)"); plt.title("Indian races: where athletes are relatively faster or slower"); plt.show()
        finding(f"At the same finish time, athletes at Indian races are relatively fastest on {weighted.idxmin()} "
                f"({weighted.min():+.0f}%) and slowest on {weighted.idxmax()} ({weighted.max():+.0f}%).")
    print(f"Indian races in data: {sorted(df.loc[df['india'], 'location'].astype(str).str.title().unique())}")
else:
    print("Too few results from Indian races.")

## 11. Repeat athletes: what improves, and how fast?

For athletes who raced more than once, we compare their first and latest race. Times are relative to each race's median (so venue differences cancel out).

**Caveat — regression to the mean:** an unusually bad day in the first race looks like improvement in the second, which inflates all these numbers. Compare segments with each other, not as absolutes.

In [ ]:
rep_path = find("repeat_athletes.csv")
if rep_path:
    rep = pd.read_csv(rep_path)
    rep["gain_pct"] = (rep["total_time_rel_a"] - rep["total_time_rel_b"]) * 100
    print(f"{len(rep):,} repeat athletes · {(rep.gain_pct > 0).mean():.0%} improved relative to their field")

    # how much of a weakness closes, per segment
    rows = []
    for c in SEGS:
        wa = rep[f"{c}_rel_a"] - rep["total_time_rel_a"]
        wb = rep[f"{c}_rel_b"] - rep["total_time_rel_b"]
        weak = wa > 0.10
        rows.append({"segment": LABEL[c], "athletes weak here": int(weak.sum()),
                     "weakness closed": ((wa - wb) / wa)[weak].clip(-1, 1.5).median()})
    close = pd.DataFrame(rows).set_index("segment").sort_values("weakness closed")
    close["weakness closed"].plot(kind="barh", color="#2e9d5b")
    plt.xlabel("Median share of the weakness closed by the next race"); plt.title("Which weaknesses actually close?"); plt.show()
    finding(f"Weaknesses close most on {close.index[-1]} ({close['weakness closed'].iloc[-1]:.0%}) "
            f"and least on {close.index[0]} ({close['weakness closed'].iloc[0]:.0%}).")

    # improvement vs time between races
    if rep["months_between"].notna().sum() >= 100:
        span, unit = rep["months_between"], "months"
        bins, labels_ = [0, 3, 6, 9, 12, 24, 999], ["≤3", "4–6", "7–9", "10–12", "13–24", "24+"]
    else:
        span, unit = (rep["year_b"] - rep["year_a"]), "years"
        bins, labels_ = [-1, 0, 1, 2, 99], ["same year", "1", "2", "3+"]
    t = rep.groupby(pd.cut(span, bins, labels=labels_), observed=True)["gain_pct"].agg(["median", "size"])
    t.columns = ["median improvement (% of field median)", "athletes"]
    display(t.round(1))
    finding(f"Improvement by {unit} between races: " + ", ".join(f"{i}: {r.iloc[0]:+.1f}%" for i, r in t.iterrows()) + ".")
else:
    print("repeat_athletes.csv not found.")

## Summary of findings

In [ ]:
for i, f in enumerate(FINDINGS, 1):
    print(f"{i}. {f}")

### Limitations
- Results data shows associations, not causes.
- Athlete identity for repeat races is matched by name within gender and division; namesakes can be mixed up.
- Venue comparisons also reflect who races where, not only the course.
- Seasons covered: see the top of the notebook.

*Want this for your own race? Try **Next Five**.*